In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

df = pd.read_csv('../data/raw/Atlantic_Spain.csv')
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')

print(f"Rows: {len(df)}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique dates: {df['date'].nunique()}")
df.head()

Rows: 27800
Date range: 2024-05-18 to 2025-11-27
Unique dates: 555


,date,position,song,artist,popularity,duration_ms,album_type,total_tracks,is_explicit,album_cover_url
0,2024-05-18,1,Supernova,Saiko,84,190909,single,1,False,https://i.scdn.co/image/ab67616d0000b273c3f5b9...
1,2024-05-18,2,EL TONTO,Lola Indigo,83,187986,album,11,False,https://i.scdn.co/image/ab67616d0000b273daab36...
2,2024-05-18,3,Acróstico,Shakira,81,170785,single,1,False,https://i.scdn.co/image/ab67616d0000b273bd6bbd...
3,2024-05-18,4,BESO,ROSALÍA & Rauw Alejandro,96,194543,single,3,False,https://i.scdn.co/image/ab67616d0000b2734d6cf0...
4,2024-05-18,5,El Merengue,Marshmello & Manuel Turizo,91,189668,single,1,False,https://i.scdn.co/image/ab67616d0000b273f40467...


In [2]:
print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isna().sum())

Shape: (27800, 10)

Column types:
date               datetime64[us]
position                    int64
song                          str
artist                        str
popularity                  int64
duration_ms                 int64
album_type                    str
total_tracks                int64
is_explicit                  bool
album_cover_url               str
dtype: object

Missing values per column:
date               0
position           0
song               0
artist             0
popularity         0
duration_ms        0
album_type         0
total_tracks       0
is_explicit        0
album_cover_url    0
dtype: int64


In [3]:
rows_per_day = df.groupby('date').size()
print("Row count distribution per day:")
print(rows_per_day.value_counts())

anomalous_days = rows_per_day[rows_per_day != 50]
print("\nDays that DON'T have exactly 50 rows:")
print(anomalous_days)

Row count distribution per day:
50     554
100      1
Name: count, dtype: int64

Days that DON'T have exactly 50 rows:
date
2025-03-01    100
dtype: int64


In [4]:
expected_days = (df['date'].max() - df['date'].min()).days + 1
actual_days = df['date'].nunique()

print(f"Expected calendar days in range: {expected_days}")
print(f"Actual unique dates in data: {actual_days}")
print(f"Gap: {expected_days - actual_days} days missing")

Expected calendar days in range: 559
Actual unique dates in data: 555
Gap: 4 days missing


In [5]:
raw_unique_songs = df['song'].nunique()
normalized_unique_songs = df['song'].str.strip().str.lower().nunique()

raw_unique_artists = df['artist'].nunique()
normalized_unique_artists = df['artist'].str.strip().str.lower().nunique()

print(f"Unique song titles (raw): {raw_unique_songs}")
print(f"Unique song titles (normalized): {normalized_unique_songs}")
print(f"Difference: {raw_unique_songs - normalized_unique_songs}")

print(f"\nUnique artists (raw): {raw_unique_artists}")
print(f"Unique artists (normalized): {normalized_unique_artists}")
print(f"Difference: {raw_unique_artists - normalized_unique_artists}")

Unique song titles (raw): 545
Unique song titles (normalized): 538
Difference: 7

Unique artists (raw): 306
Unique artists (normalized): 301
Difference: 5


Let's actually see these, so you have concrete examples instead of just a count — this is the kind of detail that makes your explanation land well if someone asks you to show them.

## See the actual case-variant songs and artists

In [6]:
# Find songs with multiple raw casings under the same normalized key
song_variants = (
    df.assign(song_key=df['song'].str.strip().str.lower())
    .groupby('song_key')['song']
    .agg(lambda x: sorted(set(x)))
)
song_variants = song_variants[song_variants.apply(len) > 1]
print("Songs with inconsistent casing:")
print(song_variants)

Songs with inconsistent casing:
song_key
classy 101                                                                                [CLASSY 101, Classy 101]
hablame claro                                                                       [HABLAME CLARO, Hablame Claro]
hay lupita                                                                                [HAY LUPITA, Hay Lupita]
japoni (feat. camin, moncho chavea, jthyago & samueliyo baby)    [Japoni (Feat. Camin, Moncho Chavea, Jthyago &...
la baby                                                                                         [LA BABY, La Baby]
no_se_ve.mp3                                                                          [No_Se_Ve.mp3, No_se_ve.mp3]
supernova                                                                                   [SUPERNOVA, Supernova]
Name: song, dtype: object


In [7]:
# Same for artists
artist_variants = (
    df.assign(artist_key=df['artist'].str.strip().str.lower())
    .groupby('artist_key')['artist']
    .agg(lambda x: sorted(set(x)))
)
artist_variants = artist_variants[artist_variants.apply(len) > 1]
print("\nArtists with inconsistent casing:")
print(artist_variants)



Artists with inconsistent casing:
artist_key
jhayco                                                                     [JHAYCO, Jhayco]
ovy on the drums & myke towers & saiko    [Ovy On The Drums & Myke Towers & SAIKO, Ovy O...
quevedo & saiko                                          [Quevedo & SAIKO, Quevedo & Saiko]
saiko                                                                        [SAIKO, Saiko]
saiko & feid & quevedo                     [SAIKO & Feid & Quevedo, Saiko & Feid & Quevedo]
Name: artist, dtype: object


In [8]:
song_key = df['song'].str.strip().str.lower()
artist_key = df['artist'].str.strip().str.lower()

metadata_cols = ['duration_ms', 'album_type', 'total_tracks', 'is_explicit']

nunique_per_song = (
    df.assign(song_key=song_key, artist_key=artist_key)
    .groupby(['song_key', 'artist_key'])[metadata_cols]
    .nunique()
)

inconsistent = nunique_per_song[(nunique_per_song > 1).any(axis=1)]
print(f"Song+artist pairs with inconsistent metadata: {len(inconsistent)}")
print(inconsistent)

Song+artist pairs with inconsistent metadata: 29
                                                                                           duration_ms  album_type  total_tracks  is_explicit
song_key                                           artist_key                                                                                
beautiful things                                   benson boone                                      1           2             2            1
coco loco                                          maluma                                            1           2             2            1
cuando te vi | crossover #5                        big one & maria becerra & trueno                  1           1             2            1
despechá                                           rosalía                                           1           1             1            2
duro                                               quevedo                                         

## xxxxx

In [9]:
explicit_conflicts = inconsistent[inconsistent['is_explicit'] > 1]
print(f"Songs with conflicting is_explicit flag: {len(explicit_conflicts)}")

conflict_keys = explicit_conflicts.index.tolist()
detail = df.assign(song_key=song_key, artist_key=artist_key)
for sk, ak in conflict_keys:
    rows = detail[(detail['song_key']==sk) & (detail['artist_key']==ak)][['date','is_explicit','album_type','total_tracks']].sort_values('date')
    print(f"\n{sk} — {ak}")
    print(rows.to_string(index=False))

Songs with conflicting is_explicit flag: 5

despechá — rosalía
      date  is_explicit album_type  total_tracks
2024-05-22        False      album            24
2024-05-23        False      album            24
2024-05-24        False      album            24
2024-05-29        False      album            24
2024-05-30        False      album            24
2024-05-31        False      album            24
2024-06-05        False      album            24
2024-06-06        False      album            24
2024-06-07        False      album            24
2024-06-08        False      album            24
2024-06-10        False      album            24
2024-06-11        False      album            24
2024-06-12        False      album            24
2024-06-13        False      album            24
2024-06-14        False      album            24
2024-06-15        False      album            24
2024-06-16        False      album            24
2024-06-17        False      album            24
2024-0

# Data preprocessing

In [10]:
before = len(df)
df = df.drop_duplicates(subset=['date', 'position'], keep='first').reset_index(drop=True)
after = len(df)

print(f"Rows before: {before}")
print(f"Rows dropped: {before - after}")
print(f"Rows after: {after}")

check = df.groupby('date').size()
print(f"Days with != 50 rows now: {(check != 50).sum()}")

Rows before: 27800
Rows dropped: 50
Rows after: 27750
Days with != 50 rows now: 0


In [11]:
full_range = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
missing_dates = sorted(set(full_range) - set(df['date'].unique()))

print(f"Missing dates ({len(missing_dates)}):")
for d in missing_dates:
    print(" -", d.date())

Missing dates (4):
 - 2025-03-14
 - 2025-03-25
 - 2025-07-11
 - 2025-08-13


In [12]:
missing_dates_df = pd.DataFrame({'missing_date': [d.date() for d in missing_dates]})
missing_dates_df.to_csv('../data/processed/known_gap_dates.csv', index=False)
print("Saved known_gap_dates.csv")

Saved known_gap_dates.csv


In [13]:
df['song_key'] = df['song'].str.strip().str.lower()
df['artist_key'] = df['artist'].str.strip().str.lower()

print(f"Unique song titles (raw): {df['song'].nunique()}")
print(f"Unique song_key (normalized): {df['song_key'].nunique()}")
print(f"Unique artists (raw): {df['artist'].nunique()}")
print(f"Unique artist_key (normalized): {df['artist_key'].nunique()}")

Unique song titles (raw): 545
Unique song_key (normalized): 538
Unique artists (raw): 306
Unique artist_key (normalized): 301


In [14]:
metadata_cols = ['duration_ms', 'album_type', 'total_tracks', 'is_explicit', 'album_cover_url']

# Get each song's most recent row, by date, to use as the canonical metadata source
canonical = (
    df.sort_values('date')
    .groupby(['song_key', 'artist_key'])
    .tail(1)[['song_key', 'artist_key'] + metadata_cols]
    .rename(columns={c: f'{c}_canonical' for c in metadata_cols})
)

df = df.merge(canonical, on=['song_key', 'artist_key'], how='left')

# Overwrite the original metadata columns with the canonical values
for c in metadata_cols:
    df[c] = df[f'{c}_canonical']
    df.drop(columns=[f'{c}_canonical'], inplace=True)

print("Metadata standardized to each song's most recent appearance.")
print(f"Shape after merge: {df.shape}")

Metadata standardized to each song's most recent appearance.
Shape after merge: (27750, 12)


In [15]:
checks = {
    "Exactly 50 rows per day": (df.groupby('date').size() == 50).all(),
    "No missing values": df.isna().sum().sum() == 0,
    "Positions 1-50 only": df['position'].between(1, 50).all(),
    "No duplicate (date, position)": not df.duplicated(subset=['date', 'position']).any(),
    "No duplicate (date, song_key, artist_key)": not df.duplicated(subset=['date', 'song_key', 'artist_key']).any(),
    "No metadata inconsistency remaining": (
        df.groupby(['song_key','artist_key'])[['duration_ms','album_type','total_tracks','is_explicit']]
        .nunique().gt(1).any(axis=1).sum() == 0
    ),
}

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} - {check}")

PASS - Exactly 50 rows per day
PASS - No missing values
PASS - Positions 1-50 only
PASS - No duplicate (date, position)
PASS - No duplicate (date, song_key, artist_key)
PASS - No metadata inconsistency remaining


In [16]:
df.to_csv('../data/processed/spain_top50_clean.csv', index=False)
print(f"Saved {len(df)} rows to data/processed/spain_top50_clean.csv")
df.head()

Saved 27750 rows to data/processed/spain_top50_clean.csv


,date,position,song,artist,popularity,duration_ms,album_type,total_tracks,is_explicit,album_cover_url,song_key,artist_key
0,2024-05-18,1,Supernova,Saiko,84,190909,album,16,False,https://i.scdn.co/image/ab67616d0000b273e346fc...,supernova,saiko
1,2024-05-18,2,EL TONTO,Lola Indigo,83,187986,album,11,False,https://i.scdn.co/image/ab67616d0000b273daab36...,el tonto,lola indigo
2,2024-05-18,3,Acróstico,Shakira,81,170785,single,1,False,https://i.scdn.co/image/ab67616d0000b273bd6bbd...,acróstico,shakira
3,2024-05-18,4,BESO,ROSALÍA & Rauw Alejandro,96,194543,single,3,False,https://i.scdn.co/image/ab67616d0000b2734d6cf0...,beso,rosalía & rauw alejandro
4,2024-05-18,5,El Merengue,Marshmello & Manuel Turizo,91,189668,single,1,False,https://i.scdn.co/image/ab67616d0000b273f40467...,el merengue,marshmello & manuel turizo
